# 🇻🇳 VietNerm — Train & Publish Pipeline

Notebook này thực hiện toàn bộ pipeline:

```
Clone repo → Install deps → Generate data → Train (GPU) → Evaluate → Push to 🤗 HuggingFace Hub
```

**Yêu cầu:** Chạy trên Google Colab với GPU runtime (Runtime → Change runtime type → T4 GPU)

## ⚙️ Cấu hình

In [ ]:
# ============================================================
# CẤU HÌNH — CHỈNH SỬA Ở ĐÂY
# ============================================================

# HuggingFace username hoặc organization
HF_USERNAME = "ngocthanhdoan"  # @param {type:"string"}

# Số lượng samples để generate (nhiều hơn = model tốt hơn, nhưng train lâu hơn)
DATASET_SIZE = 50000  # @param {type:"integer"}

# Số epochs train (khuyến nghị 5-10 với GPU)
TRAIN_EPOCHS = 5  # @param {type:"integer"}

# Batch size (T4 16GB: 16-32, A100: 32-64)
BATCH_SIZE = 16  # @param {type:"integer"}

# Doc types cần train (để trống = tất cả từ registry)
# Ví dụ: ["cccd"] hoặc ["cccd", "giay_ra_vien"]
DOC_TYPES = []  # @param {type:"raw"}

# Repo GitHub
GITHUB_REPO = "https://github.com/Devhub-Solutions/VietNerm.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"  # @param {type:"string"}

# Push private repos lên HuggingFace?
HF_PRIVATE = False  # @param {type:"boolean"}

# F1 tối thiểu để cho phép publish (quality gate)
MIN_F1 = 0.5  # @param {type:"number"}

## 1. Setup môi trường

In [ ]:
import os
import subprocess
import sys
import time
import json
from pathlib import Path
from datetime import datetime

# Kiểm tra GPU
gpu_available = False
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
    if result.returncode == 0:
        gpu_info = result.stdout.strip()
        gpu_available = True
        print(f"✅ GPU: {gpu_info}")
    else:
        print("⚠️  Không tìm thấy GPU — train sẽ chạy trên CPU (rất chậm)")
        print("   Khuyến nghị: Runtime → Change runtime type → T4 GPU")
except FileNotFoundError:
    print("⚠️  Không tìm thấy GPU — train sẽ chạy trên CPU (rất chậm)")

print(f"Python: {sys.version}")
print(f"Thời gian bắt đầu: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Clone repository
PROJECT_DIR = "/content/VietNerm"

if os.path.exists(PROJECT_DIR):
    print(f"📁 Repo đã tồn tại, đang pull latest...")
    !cd {PROJECT_DIR} && git pull origin {GITHUB_BRANCH}
else:
    print(f"📥 Cloning {GITHUB_REPO}...")
    !git clone -b {GITHUB_BRANCH} {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"📂 Working directory: {os.getcwd()}")

In [ ]:
# Cài dependencies
print("📦 Cài đặt dependencies...")
!pip install -q -r requirements.txt accelerate
print("✅ Dependencies đã cài xong")

In [ ]:
# Đăng nhập HuggingFace
from huggingface_hub import login, HfApi

# Thử dùng Colab secrets trước, nếu không có thì hỏi nhập tay
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("🔑 Đã lấy HF_TOKEN từ Colab Secrets")
except Exception:
    pass

if not hf_token:
    print("Nhập HuggingFace token (lấy từ https://huggingface.co/settings/tokens):")
    from getpass import getpass
    hf_token = getpass("HF_TOKEN: ")

login(token=hf_token)
api = HfApi(token=hf_token)
print(f"✅ Đã đăng nhập HuggingFace: {api.whoami()['name']}")

## 2. Discover doc types

In [ ]:
import yaml

# Đọc registry
with open("registry/documents.yaml") as f:
    registry = yaml.safe_load(f)

all_doc_types = list(registry["documents"].keys())

# Nếu user chỉ định doc types cụ thể, validate và dùng nó
if DOC_TYPES:
    invalid = [d for d in DOC_TYPES if d not in all_doc_types]
    if invalid:
        print(f"❌ Doc types không hợp lệ: {invalid}")
        print(f"   Có sẵn: {all_doc_types}")
        raise ValueError(f"Invalid doc types: {invalid}")
    doc_types = DOC_TYPES
else:
    doc_types = all_doc_types

print(f"📋 Sẽ train {len(doc_types)} doc type(s):")
for dt in doc_types:
    info = registry["documents"][dt]
    print(f"   • {dt} — {info['name']}")

## 3. Generate Data → Train → Evaluate → Publish

Chạy toàn bộ pipeline cho từng doc type.

In [ ]:
results = {}
total_start = time.time()

for i, doc_type in enumerate(doc_types, 1):
    doc_name = registry["documents"][doc_type]["name"]
    print(f"\n{'='*60}")
    print(f"[{i}/{len(doc_types)}] {doc_type} — {doc_name}")
    print(f"{'='*60}")

    doc_start = time.time()
    doc_result = {"doc_type": doc_type, "name": doc_name, "status": "pending"}

    try:
        # ── Step 1: Generate data ──────────────────────────────
        print(f"\n📊 [1/4] Generating {DATASET_SIZE:,} samples...")
        ret = subprocess.run(
            [sys.executable, "synthetic/generate_dataset.py",
             "--doc", doc_type, "--size", str(DATASET_SIZE)],
            capture_output=True, text=True
        )
        if ret.returncode != 0:
            print(f"❌ Generate failed:\n{ret.stderr}")
            doc_result["status"] = "generate_failed"
            doc_result["error"] = ret.stderr
            results[doc_type] = doc_result
            continue
        print(ret.stdout.split("\n")[-3])  # Dòng tóm tắt
        print("✅ Generate done")

        # ── Step 2: Train ─────────────────────────────────────
        print(f"\n🏋️ [2/4] Training ({TRAIN_EPOCHS} epochs, batch_size={BATCH_SIZE})...")
        train_cmd = [
            sys.executable, "training/train.py",
            "--doc", doc_type,
            "--epochs", str(TRAIN_EPOCHS),
            "--batch_size", str(BATCH_SIZE),
        ]
        # Stream output trực tiếp để thấy progress
        ret = subprocess.run(train_cmd)
        if ret.returncode != 0:
            print(f"❌ Training failed!")
            doc_result["status"] = "train_failed"
            results[doc_type] = doc_result
            continue
        print("✅ Training done")

        # ── Step 3: Evaluate ──────────────────────────────────
        print(f"\n🔍 [3/4] Evaluating model (min F1={MIN_F1})...")
        model_dir = f"models/phobert/{doc_type}"
        dataset_dir = f"datasets/ner/{doc_type}"

        # Đọc F1 từ trainer_state.json
        best_f1 = 0.0
        trainer_state_path = Path(model_dir) / "trainer_state.json"
        if trainer_state_path.exists():
            with open(trainer_state_path) as f:
                state = json.load(f)
            for entry in state.get("log_history", []):
                f1 = entry.get("eval_f1", 0.0)
                if f1 > best_f1:
                    best_f1 = f1

        doc_result["f1"] = best_f1
        print(f"   Best F1: {best_f1:.4f}")

        if best_f1 < MIN_F1:
            print(f"⚠️  F1 ({best_f1:.4f}) < threshold ({MIN_F1}) — bỏ qua publish")
            doc_result["status"] = "low_f1"
            results[doc_type] = doc_result
            continue
        print("✅ Evaluation passed")

        # ── Step 4: Publish to HuggingFace ────────────────────
        print(f"\n🚀 [4/4] Publishing to HuggingFace Hub...")
        model_repo = f"{HF_USERNAME}/phobert-{doc_type}-ner"
        dataset_repo = f"{HF_USERNAME}/vietnerm-{doc_type}-dataset"
        private_flag = ["--private"] if HF_PRIVATE else []

        # Push model
        print(f"   Model  → {model_repo}")
        ret = subprocess.run(
            [sys.executable, "huggingface/push_model.py",
             "--doc", doc_type, "--repo", model_repo,
             "--token", hf_token] + private_flag,
            capture_output=True, text=True
        )
        if ret.returncode != 0:
            print(f"❌ Push model failed:\n{ret.stderr}")
            doc_result["status"] = "publish_failed"
            doc_result["error"] = ret.stderr
            results[doc_type] = doc_result
            continue

        # Push dataset
        print(f"   Dataset → {dataset_repo}")
        ret = subprocess.run(
            [sys.executable, "huggingface/push_dataset.py",
             "--doc", doc_type, "--repo", dataset_repo,
             "--token", hf_token] + private_flag,
            capture_output=True, text=True
        )
        if ret.returncode != 0:
            print(f"⚠️  Push dataset failed (model vẫn đã publish):\n{ret.stderr}")

        doc_result["status"] = "published"
        doc_result["model_url"] = f"https://huggingface.co/{model_repo}"
        doc_result["dataset_url"] = f"https://huggingface.co/datasets/{dataset_repo}"
        print(f"✅ Published!")
        print(f"   🤗 Model:   https://huggingface.co/{model_repo}")
        print(f"   🤗 Dataset: https://huggingface.co/datasets/{dataset_repo}")

    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        doc_result["status"] = "error"
        doc_result["error"] = str(e)

    doc_result["time_minutes"] = round((time.time() - doc_start) / 60, 1)
    results[doc_type] = doc_result
    print(f"\n⏱️  {doc_type}: {doc_result['time_minutes']} phút")

## 4. Tổng kết

In [ ]:
total_minutes = round((time.time() - total_start) / 60, 1)

print("\n" + "=" * 60)
print("📊 KẾT QUẢ PIPELINE")
print("=" * 60)
print(f"Tổng thời gian: {total_minutes} phút")
print(f"Dataset size:   {DATASET_SIZE:,} samples")
print(f"Train epochs:   {TRAIN_EPOCHS}")
print(f"GPU:            {'Có' if gpu_available else 'Không (CPU)'}")
print()

# Bảng kết quả
print(f"{'Doc Type':<25} {'Status':<15} {'F1':>8} {'Time':>8} {'Links'}")
print("-" * 90)

for doc_type, r in results.items():
    status_icon = {
        "published": "✅",
        "low_f1": "⚠️",
        "generate_failed": "❌",
        "train_failed": "❌",
        "publish_failed": "❌",
        "error": "❌",
    }.get(r["status"], "❓")

    f1_str = f"{r.get('f1', 0):.4f}" if r.get("f1") else "—"
    time_str = f"{r.get('time_minutes', 0)}m"
    link = r.get("model_url", "")

    print(f"{status_icon} {doc_type:<23} {r['status']:<15} {f1_str:>8} {time_str:>8} {link}")

published = [r for r in results.values() if r["status"] == "published"]
print(f"\n🎉 {len(published)}/{len(results)} models published thành công!")